# Preparación del Dataset: Credit Approval

## 1. Descripción del Dataset
Datos de aprobación de tarjetas de crédito. Muchas de las variables han sido ofuscadas (escondidas) por confidencialidad (A1, A2, etc.).

Aquí procesaremos y prepararemos los datos matemáticamente desde cero usando fórmulas puras, tal como se hace en el curso. Explicaremos paso a paso el código.


## 2. Importación de Librerías
Vamos a cargar las herramientas necesarias, usando lo más básico posible:

- `pandas` y `numpy`: para leer el archivo y manipular la tabla matemáticamente.
- `matplotlib.pyplot` y `seaborn`: para hacer gráficos simples en 2D (líneas, barras), nada en 3D para no complicarlo.

*Nota:* No usaremos funciones complejas o cajas negras para normalizar ni para dividir los datos, todo lo haremos "a mano" con simples matemáticas.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NO usamos sklearn para transformar ni dividir datos.


## 3. Cargar y Explorar el Archivo (Dataset)
Aquí leemos el archivo base y mostramos cómo se ve. Revisamos la cantidad de filas y columnas que tenemos para entender su tamaño.


In [2]:
ruta = 'REPOSITORIOS_PARCIAL/6_Credit Approval – UCI/credit+approval/crx.data'

tabla_datos = pd.read_csv(ruta, header=None, na_values='?')

# Mostramos tamaño (filas, columnas)
print('Tamaño del dataset:', tabla_datos.shape)

# Vemos las primeras 5 filas
tabla_datos.head()

# Imprimir información detallada del dataset
filas, columnas = tabla_datos.shape
print(f"\n--- INFORMACIÓN DEL DATASET ---")
print(f"Total de Filas (Ejemplos): {filas}")
print(f"Total de Columnas (Características): {columnas}\n")


Tamaño del dataset: (690, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,202.0,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,43.0,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,280.0,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,100.0,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,120.0,0,+


## 4. Análisis de Datos Vacíos o Faltantes
En la vida real, los datos no siempre vienen perfectos, hay casillas vacías (Nulos o NaN). 

En este caso, hemos decidido **eliminar la fila completa** si detectamos que tiene algún dato faltante. Esto asegura que la red neuronal solo aprenda de casos reales y 100% verídicos (no inventamos promedios ni asumimos valores vacíos), a costa de reducir la cantidad de ejemplos disponibles.


In [3]:
datos_nulos = tabla_datos.isnull().sum()
print('Columnas con casillas vacías antes de limpieza:')
print(datos_nulos[datos_nulos > 0])

# Eliminamos absolutamente cualquier fila que contenga un dato nulo (NaN)
filas_antes = len(tabla_datos)
tabla_datos = tabla_datos.dropna()
filas_despues = len(tabla_datos)

print(f"\nSe eliminaron {filas_antes - filas_despues} filas que contenían datos nulos.")
print('Nulos restantes sumados:', tabla_datos.isnull().sum().sum())


Columnas con casillas vacías:
0     12
1     12
3      6
4      6
5      9
6      9
13    13
dtype: int64

Nulos restantes sumados: 0


C:\Users\alfao\AppData\Local\Temp\ipykernel_8216\885227427.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = tabla_datos.select_dtypes(include=['object']).columns


## 5. Conversión de Textos a Números
Como la computadora o nuestra Red Neuronal no entiende palabras , necesitamos números. Usaremos una herramienta integrada muy sencilla llamada `get_dummies`.

Lo que hace es tomar las columnas de texto y crear columnas visuales llenas de ceros (`0`) y unos (`1`). Es directo y no necesitas complicarlo con diccionarios o For loops.


In [4]:
# Convertimos todo el texto a datos de 0 y 1 de forma directa y sencilla.
tabla_datos = pd.get_dummies(tabla_datos, dtype=int)

# Mostramos cómo quedó la tabla ahora, lista solo con formato numérico
tabla_datos.head()


,1,2,7,10,13,14,0_Desconocido,0_a,0_b,3_Desconocido,...,8_t,9_f,9_t,11_f,11_t,12_g,12_p,12_s,15_+,15_-
0,30.83,0.000,1.25,1,202.0,0,0,0,1,0,...,1,0,1,1,0,1,0,0,1,0
1,58.67,4.460,3.04,6,43.0,560,0,1,0,0,...,1,0,1,1,0,1,0,0,1,0
2,24.50,0.500,1.50,0,280.0,824,0,1,0,0,...,1,1,0,1,0,1,0,0,1,0
3,27.83,1.540,3.75,5,100.0,3,0,0,1,0,...,1,0,1,0,1,1,0,0,1,0
4,20.17,5.625,1.71,0,120.0,0,0,0,1,0,...,1,1,0,1,0,0,0,1,1,0


## 6. Normalización de características (Feature Normalize)
Al visualizar los datos se puede observar que las características tienen diferentes magnitudes (por ejemplo, precios altísimos y números pequeños).

**La fórmula para calcular esto con escalado Z-Score es:**
$$ X_{norm} = \frac{X - \mu}{\sigma} $$

En código lo aplicamos usando las operaciones directas `mean()` (media) y `std()` (desviación estándar).


In [5]:
# 1. Calculamos la media (mu) y desviación estándar (sigma) para cada una de las columnas
mu = tabla_datos.mean()
sigma = tabla_datos.std()

# Para evitar errores por dividir entre cero (si la desviación es 0 en alguna columna), forzamos que sea mínimo 1
sigma_seguro = np.where(sigma == 0, 1, sigma)

# 2. Aplicamos la fórmula matemática exacta: (X - mu) / sigma
datos_normalizados = (tabla_datos - mu) / sigma_seguro

# Mostrar la tabla final convertida
datos_normalizados.head()


,1,2,7,10,13,14,0_Desconocido,0_a,0_b,3_Desconocido,...,8_t,9_f,9_t,11_f,11_t,12_g,12_p,12_s,15_+,15_-
0,-0.062276,-0.955920,-0.290872,-0.287892,0.104469,-0.195272,-0.132942,-0.660958,0.688238,-0.093591,...,0.953958,-1.156306,1.156306,0.918529,-0.918529,0.322257,-0.108228,-0.299861,1.116131,-1.116131
1,2.286443,-0.060007,0.244013,0.740293,-0.819095,-0.087788,-0.132942,1.510762,-1.450880,-0.093591,...,0.953958,-1.156306,1.156306,0.918529,-0.918529,0.322257,-0.108228,-0.299861,1.116131,-1.116131
2,-0.596305,-0.855481,-0.216167,-0.493529,0.557537,-0.037117,-0.132942,1.510762,-1.450880,-0.093591,...,0.953958,0.863570,-0.863570,0.918529,-0.918529,0.322257,-0.108228,-0.299861,1.116131,-1.116131
3,-0.315370,-0.646569,0.456175,0.534656,-0.488006,-0.194696,-0.132942,-0.660958,0.688238,-0.093591,...,0.953958,-1.156306,1.156306,-1.087120,1.087120,0.322257,-0.108228,-0.299861,1.116131,-1.116131
4,-0.961605,0.174015,-0.153415,-0.493529,-0.371835,-0.195272,-0.132942,-0.660958,0.688238,-0.093591,...,0.953958,0.863570,-0.863570,0.918529,-0.918529,-3.098621,-0.108228,3.330040,1.116131,-1.116131


## 7. Dividir el Dataset en 80/20 (Entrenamiento y Prueba)
Finalmente apartaremos el 80% de nuestros datos para que el modelo aprenda y un 20% para hacerle un examen y evaluarlo.
En lugar de una caja negra, apartamos las filas simplemente cortando las tablas basándonos en el porcentaje.


In [6]:
# Extraemos la última columna para usarla como objetivo de predicción (Variable Y o Target)
objetivo_col = datos_normalizados.columns[-1]
caracteristicas = datos_normalizados.drop(objetivo_col, axis=1)
objetivo = datos_normalizados[objetivo_col]

# --------- DIVISIÓN A MANO (80% Entrenamiento, 20% Prueba) ---------

# 1. Calculamos cuánto es el 80% entero de nuestras filas
total_filas = len(datos_normalizados)
ochenta_por_ciento = int(total_filas * 0.8)

# 2. Cortamos las tablas usando posiciones desde 0 hasta el 80%
X_entrenamiento = caracteristicas.iloc[:ochenta_por_ciento]
y_entrenamiento = objetivo.iloc[:ochenta_por_ciento]

# 3. Y para el resto (el 20% restante), usamos desde el 80% en adelante
X_prueba = caracteristicas.iloc[ochenta_por_ciento:]
y_prueba = objetivo.iloc[ochenta_por_ciento:]

print('Total datos Original:', total_filas, 'filas')
print('Datos para Entrenar (80%):', len(X_entrenamiento), 'filas')
print('Datos para Probar (20%):', len(X_prueba), 'filas')
print('\n¡El Dataset está preparado puramente usando lógica matemática!')


Total datos Original: 690 filas
Datos para Entrenar (80%): 552 filas
Datos para Probar (20%): 138 filas

¡El Dataset está preparado puramente usando lógica matemática!
